# Lab 2: Vector Databases with Qdrant

Welcome to Lab 2! In this session, we'll learn how to use **vector databases** to enable powerful semantic search capabilities for your AI applications.

**What you'll learn:**
- What vector databases are and why they matter for AI
- How to convert text into embeddings (vector representations)
- How to store and search vectors using Qdrant
- How to filter search results using metadata
- Building blocks for Retrieval Augmented Generation (RAG)

**Prerequisites:**
- Completed Lab 1 (basic LLM concepts)
- Basic Python and Pandas knowledge
- A local Python environment (hosted Qdrant is optional)

**References:**
- [Qdrant Beginner Tutorial](https://qdrant.tech/documentation/beginner-tutorials/search-beginners/)
- [Qdrant Cloud Quickstart](https://qdrant.tech/documentation/cloud-quickstart/)

---


## Understanding Key Concepts

### What is a Vector Database?

A **vector database** is a specialized database designed to store and search **embeddings** (numerical representations of data). Unlike traditional databases that search for exact matches, vector databases find items that are **semantically similar**.

**Example:**
- Traditional search: "diet soda" only finds items with those exact words
- Vector search: "diet soda" also finds "zero calorie beverage", "sugar-free drink", etc.

### What are Embeddings?

**Embeddings** are numerical representations (arrays of numbers) that capture the **meaning** of text. Similar texts have similar embeddings.

```
"I love cats" → [0.12, -0.34, 0.56, ...]  (384 numbers)
"I adore kittens" → [0.11, -0.33, 0.55, ...]  (similar numbers!)
"I hate Mondays" → [-0.45, 0.22, -0.18, ...]  (different numbers)
```

### What is Similarity Search?

**Similarity search** finds items in the database whose embeddings are closest to a query embedding. This is measured using distance metrics like **cosine similarity**.

### Why Qdrant?

**Qdrant** is an open-source vector database that:
-  Is fast and scalable
-  Supports metadata filtering
-  Has a generous free cloud tier (On the cloud hosted platform)
-  Works great with Python

---


## Step 1: Environment setup

Use the pinned requirements in the README. The embedding model runs locally on CPU and downloads on its first use. Qdrant runs in memory by default, so this lab needs no cloud account.


In [ ]:
# Install requirements.txt from the README before opening this notebook.
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "lab_support.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the cloned AI_Trainings repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Repository:", ROOT.name)


Run the following cells in order. Restart the kernel if you changed the installed dependencies.


## Step 2: Load the Soft Drinks Data

Now let's load our soft drinks dataset. This CSV file contains product information including:
- Product names and descriptions
- Brand information
- Category/shelf classification
- Other metadata

We'll use this data to build a searchable product database!


In [ ]:
import os
import uuid
import pandas as pd
from qdrant_client import models
from lab_support import get_encoder, get_qdrant_client
df = pd.read_csv(ROOT / "data" / "softdrinks.csv").fillna("")
print(f"Loaded {len(df)} products")
df.head()


### Convert DataFrame to Documents

For Qdrant, we need to convert our DataFrame rows into a list of dictionaries (documents). Each document will become a point in our vector database.


In [ ]:
# Convert DataFrame to a list of dictionaries
documents = df.to_dict(orient='records')

print(f" Created {len(documents)} documents")
print("\n Example document (first product):")
documents[0]


## Step 3: Connect to Qdrant

With no `QDRANT_URL`, this is an in-memory database that disappears when the notebook exits. For an optional hosted service, set `QDRANT_URL` and `QDRANT_API_KEY` before starting Jupyter; product text and embeddings will be sent there.


In [ ]:
qdrant_client = get_qdrant_client()
print("Storage:", "hosted Qdrant" if os.getenv("QDRANT_URL") else "local, in-memory Qdrant")


---

## Step 4: Initialize the Embedding Model

We'll use **SentenceTransformers** to convert text into embeddings. The model `all-MiniLM-L6-v2` is:
- Fast and lightweight
- Creates 384-dimensional vectors
- Great for semantic similarity tasks

The first time you run this, it will download the model (~90MB).


In [ ]:
# Initialize the embedding model
# This model converts text into 384-dimensional vectors
encoder = get_encoder()

# Let's see how embeddings work with a quick example
example_text = "refreshing diet cola"
example_embedding = encoder.encode(example_text)

print(f" Embedding model loaded!")
print(f" Embedding dimension: {len(example_embedding)}")
print(f"\n Example embedding for '{example_text}':")
print(f"   First 10 values: {example_embedding[:10].round(4)}")


---

## Step 5: Create a Qdrant Collection

A **collection** in Qdrant is like a table in a traditional database. It stores all your vectors and their associated data (payload).

When creating a collection, we specify:
- **Vector size**: Must match our embedding dimension (384)
- **Distance metric**: How similarity is calculated (COSINE is most common)


In [ ]:
# A unique name prevents a rerun from overwriting anyone else's collection.
collection_name = "ai_trainings_products_" + uuid.uuid4().hex
qdrant_client.create_collection(collection_name=collection_name,
    vectors_config=models.VectorParams(size=encoder.get_sentence_embedding_dimension(),
                                      distance=models.Distance.COSINE))
print("Created:", collection_name)


---

##  Step 6: Create Metadata Indexes

**Payload indexes** allow us to efficiently filter search results by metadata fields. Without indexes, filtering would be slow on large datasets.

We'll create indexes for:
- `bpn` - Product number
- `shelf` - Product category (e.g., "Cola", "Sports Drinks")
- `brand` - Brand name
- `category` - Product category
- `product_name` - Product name


In [ ]:
index_fields = ["bpn", "shelf", "brand", "category", "product_name"]
if os.getenv("QDRANT_URL"):
    for field in index_fields:
        qdrant_client.create_payload_index(collection_name=collection_name,
            field_name=field, field_schema=models.PayloadSchemaType.KEYWORD)
    print("Hosted payload indexes created.")
else:
    print("Local Qdrant supports these filters without payload indexes.")


---

##  Step 7: Upload Documents with Embeddings

Now we'll upload our products to Qdrant. For each document:
1. Generate an embedding from the **product description**
2. Store the embedding as a vector
3. Store all product info as the **payload** (metadata)

This is where the magic happens - we're converting text descriptions into searchable vectors!


In [ ]:
vectors = encoder.encode([doc.get("description", "") for doc in documents],
                         normalize_embeddings=True)
qdrant_client.upsert(collection_name=collection_name, wait=True, points=[
    models.PointStruct(id=idx, vector=vector.tolist(), payload=doc)
    for idx, (doc, vector) in enumerate(zip(documents, vectors))])
assert qdrant_client.count(collection_name=collection_name, exact=True).count == len(documents)
print(f"Indexed {len(documents)} products.")


---

##  Step 8: Similarity Search

Now let's search our database! We'll:
1. Convert our search query into an embedding
2. Find the most similar products based on cosine similarity
3. Return the top results

**Example query**: "diet drink" - This should find low-calorie, sugar-free beverages even if they don't contain the exact words "diet drink".


In [ ]:
# Basic similarity search
query = "diet drink"

# Search for similar products
hits = qdrant_client.query_points(
    collection_name=collection_name,
    query=encoder.encode(query).tolist(),  # Convert query to embedding
    limit=3,  # Return top 3 results
).points

# Display results
print(f" Search query: '{query}'")
print(f" Found {len(hits)} results:\n")

for i, hit in enumerate(hits, 1):
    print(f"--- Result {i} (Score: {hit.score:.4f}) ---")
    print(f"   Product: {hit.payload.get('product_name', 'N/A')}")
    print(f"   Brand: {hit.payload.get('brand', 'N/A')}")
    print(f"   Shelf: {hit.payload.get('shelf', 'N/A')}")
    print(f"   Description: {hit.payload.get('description', 'N/A')[:100]}...")
    print()


### What Just Happened?

1. We converted "diet drink" into a 384-dimensional embedding
2. Qdrant compared this embedding against all product embeddings
3. It returned the products with the highest cosine similarity scores
4. Even products without the exact words "diet" or "drink" can match, we pulled brand name pepsi for example with no of those two words in the product name!

**Cosine similarity**: Ranges from -1 to 1 (higher = more similar). It is not a probability of relevance.

---

## Step 9: Filtered Similarity Search

Sometimes you want to combine semantic search with exact filters. For example:
- Find "diet drinks" but **only in the Cola category**
- Search for "refreshing" but **only Pepsi brand**

This is the power of Qdrant's metadata filtering!


In [ ]:
# Filtered similarity search
# Search for "diet drink" but ONLY in the Cola shelf

query = "diet drink"
filter_shelf = "Cola"

hits = qdrant_client.query_points(
    collection_name=collection_name,
    query=encoder.encode(query).tolist(),
    query_filter=models.Filter(
        must=[
            models.FieldCondition(
                key="shelf",
                match=models.MatchValue(value=filter_shelf)
            ),
        ]
    ),
    limit=2,  # Return top 2 results
).points

# Display results
print(f" Search query: '{query}'")
print(f" Filter: shelf = '{filter_shelf}'")
print(f" Found {len(hits)} results:\n")

for i, hit in enumerate(hits, 1):
    print(f"--- Result {i} (Score: {hit.score:.4f}) ---")
    print(f"   Product: {hit.payload.get('product_name', 'N/A')}")
    print(f"   Brand: {hit.payload.get('brand', 'N/A')}")
    print(f"   Shelf: {hit.payload.get('shelf', 'N/A')}")
    print()

assert hits and all(hit.payload["shelf"] == filter_shelf for hit in hits)


The filter enforces the shelf constraint. A high cosine score indicates similarity in this embedding space; it does not guarantee the product meets every requirement. Read the retrieved descriptions.


---

# 🧪 LAB EXERCISES

Now it's your turn! Complete the following exercises to practice what you've learned.

---

### Exercise 1: Basic Similarity Search

**Task:** Search for "refreshing summer beverage" and return the top 3 results.

Print out the product name, brand, and similarity score for each result.


In [ ]:
print("---Lab Exercise 1: Basic Similarity Search---\n")

# Your code here!


---

### Exercise 2: Filtered Search by Brand

**Task:** Search for "energy boost" but filter to only show products from a specific brand.

**Hints:**
- Use `query_filter` with a `FieldCondition` on the "brand" field
- Try filtering by brands like "Red Bull", "Monster", or "Gatorade"


In [ ]:
print("---Lab Exercise 2: Filtered Search by Brand---\n")

# Your code here!



---

### Exercise 3: Create Your Own Search! 🎮

**Task:** Write your own similarity search query against the soft drinks database.

**Ideas to try:**
- "healthy sports drink"
- "kids party drink"
- "coffee alternative"
- "natural fruit flavor"
- Filter by multiple conditions (category AND brand)

Have fun exploring!!!! and Share future sesstions with your team mates.


In [ ]:
print("---Lab Exercise 3: Your Own Search!---\n")

# ========================================
#  YOUR CODE HERE - Be creative!
# ========================================

# Example: Search with multiple filters



---

# 🎉 Congratulations!

You've completed Lab 2! Here's what you learned:

| Concept | What You Learned |
|---------|------------------|
| **Vector Databases** | How to store and search embeddings with Qdrant |
| **Embeddings** | How to convert text to numerical vectors using SentenceTransformers |
| **Similarity Search** | How to find semantically similar items |
| **Metadata Filtering** | How to combine semantic search with exact filters |
| **Payload Indexes** | How to optimize filtered queries |

##  How This Connects to RAG

What you learned today is a key building block for **Retrieval Augmented Generation (RAG)**:

1. **Store** your documents in a vector database (what we did today)
2. **Search** for relevant documents based on user queries
3. **Augment** LLM prompts with retrieved context
4. **Generate** accurate, grounded responses

>**Note** to see how your Lab Answers compare, go to the Solutions Folder to see our answers!

##  What's Next?

In the next session, we'll combine vector search with LLMs to build a complete RAG application!

## Additional Resources

- [Qdrant Documentation](https://qdrant.tech/documentation/)
- [SentenceTransformers Models](https://www.sbert.net/docs/pretrained_models.html)
- [Vector Search Explained](https://qdrant.tech/articles/vector-search/)
- [LangChain + Qdrant Integration](https://python.langchain.com/docs/integrations/vectorstores/qdrant)
